# Bank of Thailand API with settfex (1.0.0)

This needs **your own free tokens**, one per product, from https://portal.api.bot.or.th (guide:
https://portal.api.bot.or.th/manual). Put them in a `.env` file next to this notebook (see
`.env.example`), or pass `token=...` to a call. settfex never prints a token.

In [ ]:
from settfex.services.bot import (
    KNOWN_SERIES,
    BankOfThailand,
    get_bibor,
    get_exchange_rates,
    get_policy_rate,
    get_stat_observations,
    search_stat_series,
)

## The policy rate

In [ ]:
rate = await get_policy_rate()
print(rate.rate, rate.announcement_date)
print(rate.news_text_en[:200])

## BIBOR: the average fixing, and each bank's quotes

In [ ]:
bibor = await get_bibor("2026-11-16", "2026-11-20")
for row in bibor.rows:
    print(
        row.period, row.overnight, row.three_months, row.six_months, row.nine_months
    )  # '' becomes None

## Statistics: THOR (daily) and government yields (monthly)

Government T-bill and bond yields are **monthly averages**. For a daily short rate, use THOR or
BIBOR. The series search lists discontinued codes first, so check `observation_end`.

In [ ]:
thor3m = await get_stat_observations(KNOWN_SERIES["thor_average_3m"], "2026-11-01")
print(thor3m.frequency, thor3m.latest)
gov6m = await get_stat_observations(KNOWN_SERIES["government_yield_6m"], "2026-01-01")
print(gov6m.frequency, gov6m.latest, gov6m.last_update_date)
for s in (await search_stat_series("government bond yield"))[:5]:
    print(
        s.series_code, s.frequency, s.observation_start, "->", s.observation_end, s.series_name_eng
    )

## THB exchange rates

In [ ]:
usd = await get_exchange_rates("monthly", "2026-08", "2026-10", currency="USD")
print([(r.period, r.mid_rate) for r in usd.rows])

## An empty table is not data

Four Interest Rates tables have returned only empty rows since 2024-12-27. `has_values` tells you;
`len()` does not.

In [ ]:
bot = BankOfThailand()
spot = await bot.interest_rates.fetch_spot_rates("2026-11-20")
print(len(spot), spot.has_values, spot.header.is_discontinued, spot.header.last_updated)